<a href="https://colab.research.google.com/github/MamaChase/.ipynb/blob/main/Copy_of_04_Learners_Notebook_Low_Code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Problem Statement

### Business Context

The healthcare industry is rapidly evolving, with professionals facing increasing challenges in managing vast volumes of medical data while delivering accurate and timely diagnoses. The need for quick access to comprehensive, reliable, and up-to-date medical knowledge is critical for improving patient outcomes and ensuring informed decision-making in a fast-paced environment.

Healthcare professionals often encounter information overload, struggling to sift through extensive research and data to create accurate diagnoses and treatment plans. This challenge is amplified by the need for efficiency, particularly in emergencies, where time-sensitive decisions are vital. Furthermore, access to trusted, current medical information from renowned manuals and research papers is essential for maintaining high standards of care.

To address these challenges, healthcare centers can focus on integrating systems that streamline access to medical knowledge, provide tools to support quick decision-making, and enhance efficiency. Leveraging centralized knowledge platforms and ensuring healthcare providers have continuous access to reliable resources can significantly improve patient care and operational effectiveness.

**Common Questions to Answer**

1. **Critical Care Protocols:** "What is the protocol for managing sepsis in a critical care unit?"

2. **General Surgery:** "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"

3. **Dermatology:** "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"

4. **Neurology:** "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"


### Objective

As an AI specialist, your task is to develop a RAG-based AI solution using renowned medical manuals to address healthcare challenges. The objective is to **understand** issues like information overload, **apply** AI techniques to streamline decision-making, **analyze** its impact on diagnostics and patient outcomes, **evaluate** its potential to standardize care practices, and **create** a functional prototype demonstrating its feasibility and effectiveness.

### Data Description

The **Merck Manuals** are medical references published by the American pharmaceutical company Merck & Co., that cover a wide range of medical topics, including disorders, tests, diagnoses, and drugs. The manuals have been published since 1899, when Merck & Co. was still a subsidiary of the German company Merck.

The manual is provided as a PDF with over 4,000 pages divided into 23 sections.

## **Please read the instructions carefully before starting the project.**

This is a commented Python Notebook file in which all the instructions and tasks to be performed are mentioned.
* Blanks '_____' are provided in the notebook that
needs to be filled with an appropriate code to get the correct result. With every '_____' blank, there is a comment that briefly describes what needs to be filled in the blank space.
* Identify the task to be performed correctly, and only then proceed to write the required code.
* Please run the codes in a sequential manner from the beginning to avoid any unnecessary errors.
* Add the results/observations (wherever mentioned) derived from the analysis in the presentation and submit the same. Any mathematical or computational details which are a graded part of the project can be included in the Appendix section of the presentation.

## Installing and Importing Necessary Libraries and Dependencies

In [ ]:
# Install required libraries
!pip install -q langchain_community==0.3.27 \
              langchain==0.3.27 \
              chromadb==1.0.15 \
              pymupdf==1.26.3 \
              tiktoken==0.9.0 \
              datasets==4.0.0 \
              evaluate==0.4.5 \
              langchain_openai==0.3.30

**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [ ]:
# Import core libraries
import os                                                                       # Interact with the operating system (e.g., set environment variables)
import json                                                                     # Read/write JSON data
import requests  # type: ignore                                                 # Make HTTP requests (e.g., API calls); ignore type checker

# Import libraries for working with PDFs and OpenAI
from langchain_community.document_loaders import PyMuPDFLoader                  # Load and extract text from PDF files
# from langchain_community.document_loaders import PyPDFLoader                    # Load and extract text from PDF files
from openai import OpenAI                                                       # Access OpenAI's models and services

# Import libraries for processing dataframes and text
import tiktoken                                                                 # Tokenizer used for counting and splitting text for models
import pandas as pd                                                             # Load, manipulate, and analyze tabular data

# Import LangChain components for data loading, chunking, embedding, and vector DBs
from langchain.text_splitter import RecursiveCharacterTextSplitter              # Break text into overlapping chunks for processing
from langchain_openai import OpenAIEmbeddings                                   # Create vector embeddings using OpenAI's models
from langchain_community.vectorstores import Chroma                             # Store and search vector embeddings using Chroma DB

from datasets import Dataset                                                    # Used to structure the input (questions, answers, contexts etc.) in tabular format
from langchain_openai import ChatOpenAI                                         # This is needed since LLM is used in metric computation

## Question Answering using LLM

### OpenAI API Calling



> **Note:** <br> Make sure to create a `config.json` file in your project directory containing your OpenAI credentials in the following format:
<br><br>```{"API_KEY": "your_openai_api_key_here","OPENAI_API_BASE": "your_api_base"}```<br><br>
Replace the placeholder with your actual API key. This file allows your script to securely load API configuration details without hardcoding them directly into the code. </br>


In [ ]:
# Load the JSON file and extract values
file_name = "config.json"                                                       # Name of the configuration file
with open(file_name, 'r') as file:                                              # Open the config file in read mode
    config = json.load(file)                                                    # Load the JSON content as a dictionary
    OPENAI_API_KEY = config.get("OPENAI_API_KEY")                                             # Extract the API key from the config
    OPENAI_API_BASE = config.get("OPENAI_API_BASE")                             # Extract the OpenAI base URL from the config

# Store API credentials in environment variables
os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY                                          # Set API key as environment variable
os.environ["OPENAI_BASE_URL"] = OPENAI_API_BASE                                 # Set API base URL as environment variable

# Initialize OpenAI client
client = OpenAI()                                                               # Create an instance of the OpenAI client

In [ ]:
# Import the necessary library to access Colab secrets
from google.colab import userdata

# Retrieve the OpenAI API Key and Base URL from Colab secrets
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
OPENAI_API_BASE = userdata.get('OPENAI_API_BASE')

# Store API credentials in environment variables
os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY
os.environ["OPENAI_BASE_URL"] = OPENAI_API_BASE

# Initialize OpenAI client
client = OpenAI()

print("OpenAI API Key and Base URL loaded from Colab secrets and client initialized.")

OpenAI API Key and Base URL loaded from Colab secrets and client initialized.


In [ ]:
import json

config_data = {
    "OPENAI_API_KEY": "your_openai_api_key_here",
    "OPENAI_API_BASE": "your_api_base_url_here"
}

with open("config.json", "w") as f:
    json.dump(config_data, f, indent=4)

print("config.json created. Please replace placeholder values with your actual API key and base URL.")

config.json created. Please replace placeholder values with your actual API key and base URL.


### Response

In [ ]:
# Define a function to get a response
def response(user_prompt, max_tokens=128, temperature=0, top_p=0.95):   # Complete the code to set default paramenters
    # Create a chat completion using the OpenAI client
    completion = client.chat.completions.create(
        model="gpt-3.5-turbo",  # Complete the code by specifying the model to be used.
        messages=[
            {"role": "user", "content": user_prompt}                            # User prompt is the input/query to respond to
        ],
        max_tokens=max_tokens,                                                  # Max number of tokens to generate in the response
        temperature=temperature,                                                # Controls randomness in output
        top_p=top_p                                                             # Controls diversity via nucleus sampling
    )
    return completion.choices[0].message.content                                # Return the text content from the model's reply

### Question 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
question_1 = "What is the protocol for managing sepsis in a critical care unit?"
base_prompt_response_1=response(question_1)

### Question 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
question_2 = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?" #Complete the code to define the question #2
base_prompt_response_2=response(question_2) #Complete the code to pass the user input

### Question 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
question_3 = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?" #Complete the code to define the question #3
base_prompt_response_3=response(question_3) #Complete the code to pass the user input

### Question 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
question_4 = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?" #Complete the code to define the question #4
base_prompt_response_4=response(question_4) #Complete the code to pass the user input

### Storing the generated outputs from the base prompt


In [ ]:
# Create the DataFrame
result_df = pd.DataFrame({
    "questions": [question_1, question_2, question_3, question_4],
    "base_prompt_responses": [base_prompt_response_1, base_prompt_response_2, base_prompt_response_3, base_prompt_response_4]})

# Display the DataFrame
result_df.head()

,questions,base_prompt_responses
0,What is the protocol for managing sepsis in a ...,The protocol for managing sepsis in a critical...
1,"What are the common symptoms for appendicitis,...",Common symptoms of appendicitis include:\n\n1....
2,What are the effective treatments or solutions...,"Sudden patchy hair loss, also known as alopeci..."
3,What treatments are recommended for a person w...,The recommended treatments for a person who ha...


**Observations:**
-
-

## Question Answering using LLM with Prompt Engineering

In the next step, we will use prompt engineering to check the effect of a more detailed and well-engineered prompt on the output of the model.

In [ ]:
system_prompt = """
You are a helpful medical assistant. Provide clear and concise answers to medical questions based on your knowledge.
""" #Complete the code to define the system prompt

### Defining the function to Generate a Response From the LLM

In [ ]:
# Define a function to get a response from the OpenAI chat model
def response(system_prompt, user_prompt, max_tokens=128, temperature=0, top_p=0.95):  # Complete the code to set default paramenters
    # Create a chat completion using the OpenAI client
    completion = client.chat.completions.create(
        model="gpt-3.5-turbo",                                                        # Complete the code by specifying the model to be used.
        messages=[
            {"role": "system", "content": system_prompt},                       # System prompt sets the assistant's behavior
            {"role": "user", "content": user_prompt}                            # User prompt is the input/query to respond to
        ],
        max_tokens=max_tokens,                                                  # Max number of tokens to generate in the response
        temperature=temperature,                                                # Controls randomness in output (0 = deterministic)
        top_p=top_p                                                             # Controls diversity via nucleus sampling
    )
    return completion.choices[0].message.content                                # Return the text content from the model's reply

### Question 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
response_with_prompt_eng_1=response(system_prompt,question_1)
response_with_prompt_eng_1

'The protocol for managing sepsis in a critical care unit typically involves the following steps:\n\n1. Early recognition and diagnosis of sepsis using clinical criteria and laboratory tests.\n2. Prompt initiation of broad-spectrum antibiotics to target the suspected causative organism.\n3. Aggressive fluid resuscitation to restore adequate tissue perfusion and blood pressure.\n4. Vasopressor therapy may be necessary to support blood pressure if fluid resuscitation alone is insufficient.\n5. Monitoring and management of organ dysfunction, such as respiratory support for acute respiratory distress syndrome (ARDS) or renal replacement therapy for acute kidney injury.\n6. Source control,'

### Question 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
response_with_prompt_eng_2=response(system_prompt,question_2) #Complete the code to pass the user prompt and system prompt
response_with_prompt_eng_2

'Common symptoms of appendicitis include:\n\n1. Sudden pain that starts around the navel and moves to the lower right abdomen\n2. Loss of appetite\n3. Nausea and vomiting\n4. Fever\n5. Constipation or diarrhea\n6. Abdominal bloating\n\nAppendicitis is typically treated with surgery, specifically an appendectomy. This involves removing the inflamed appendix before it ruptures and causes a potentially life-threatening infection. In some cases, if the appendix has not ruptured and the infection is mild, antibiotics may be used to temporarily treat the condition. However, surgery is the definitive treatment for'

### Question 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
response_with_prompt_eng_3=response(system_prompt,question_3) #Complete the code to pass the user prompt and system prompt
response_with_prompt_eng_3

'Sudden patchy hair loss, known as alopecia areata, can be distressing but there are effective treatments available. Possible causes include autoimmune reactions, genetics, stress, and certain medical conditions. \n\nTreatment options include:\n1. Corticosteroid injections: Directly injected into the bald patches to suppress the immune response.\n2. Topical corticosteroids: Applied to the affected areas to reduce inflammation.\n3. Minoxidil: Over-the-counter medication that can help stimulate hair growth.\n4. Anthralin: Topical medication that alters immune function in the affected area.\n5. Immunotherapy: Using chemicals to'

### Question 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
response_with_prompt_eng_4=response(system_prompt,question_4) #Complete the code to pass the user prompt and system prompt
response_with_prompt_eng_4

"The treatment for a person who has sustained a physical injury to brain tissue, such as a traumatic brain injury (TBI), will depend on the severity of the injury. In general, treatment may include:\n\n1. **Monitoring and Observation:** Close monitoring of the person's condition to assess for any changes or complications.\n\n2. **Medication:** Medications may be prescribed to manage symptoms such as pain, seizures, or mood disturbances.\n\n3. **Surgery:** In some cases, surgery may be necessary to relieve pressure on the brain or to repair damaged tissue.\n\n4. **Rehabilitation:** Physical therapy, occupational therapy, speech therapy"

### Storing the generated outputs from the structured prompts

In [ ]:
# Add the results to a new column in the DataFrame
result_df['responses_with_prompt_eng'] = [response_with_prompt_eng_1, response_with_prompt_eng_2, response_with_prompt_eng_3, response_with_prompt_eng_4]

# Display the DataFrame
result_df.head()

,questions,base_prompt_responses,responses_with_prompt_eng
0,What is the protocol for managing sepsis in a ...,The protocol for managing sepsis in a critical...,The protocol for managing sepsis in a critical...
1,"What are the common symptoms for appendicitis,...",Common symptoms of appendicitis include:\n\n1....,Common symptoms of appendicitis include:\n\n1....
2,What are the effective treatments or solutions...,"Sudden patchy hair loss, also known as alopeci...","Sudden patchy hair loss, known as alopecia are..."
3,What treatments are recommended for a person w...,The recommended treatments for a person who ha...,The treatment for a person who has sustained a...


**Observations**:
-
-


## Data Preparation for RAG

### Loading the Data

In [ ]:
# uncomment and run the below code snippets if the dataset is present in the Google Drive
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [ ]:
manual_pdf_path = "merck_manual.pdf" #Complete the code to define the file name

In [ ]:
pdf_loader = PyMuPDFLoader(manual_pdf_path)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

After running the cell above and following the authentication steps to mount your Google Drive, you can copy the `merck_manual.pdf` file from your Drive to the Colab working directory. **Please replace `Your_Drive_Path/merck_manual.pdf` with the actual path to your PDF file within your Google Drive.** For example, if it's directly in 'My Drive', the path would be `/content/drive/My Drive/merck_manual.pdf`.

In [ ]:
# Replace 'Your_Drive_Path/merck_manual.pdf' with the actual path to your file in Google Drive
!cp "/content/drive/My Drive/Your_Drive_Path/merck_manual.pdf" "merck_manual.pdf"

After copying, you can verify that the file is in your Colab environment by listing the files:

In [ ]:
!ls

Once the file is copied successfully, you should restart the runtime (Runtime > Restart runtime) and then run all cells from the beginning to ensure all previous steps, including library installations and variable definitions, are correctly re-executed with the PDF now available.

In [ ]:
manual = pdf_loader.load()

### Data Overview

#### Checking the first 5 pages

In [ ]:
for i in range(5):
    print(f"Page Number : {i+1}",end="\n")
    print(manual[i].page_content,end="\n")

### Data Chunking

#### Chunk the PDF into Manageable Text Sections Using a Token-Based Splitter

In [ ]:
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name='cl100k_base',
    chunk_size=1000, #Complete the code to define the chunk size
    chunk_overlap=200 #Complete the code to define the chunk overlap
)

#### Split the Loaded PDF into Chunks for Further Processing

In [ ]:
document_chunks = pdf_loader.load_and_split(text_splitter)

#### Check the Number of Chunks Created

In [ ]:
len(document_chunks)

### Generate Vector Embeddings for Text Chunks Using OpenAI

In [ ]:
# Initialize the OpenAI Embeddings model with API credentials
embedding_model = OpenAIEmbeddings(
    openai_api_key=OPENAI_API_KEY,                                                     # Your OpenAI API key for authentication
    openai_api_base=OPENAI_API_BASE                                             # The OpenAI API base URL endpoint
)

# Generate embeddings (vector representations) for the first two document chunks
embedding_1 = embedding_model.embed_query(document_chunks[0].page_content)      # Embedding for chunk 0
embedding_2 = embedding_model.embed_query(document_chunks[1].page_content)      # Embedding for chunk 1

# Check and print the dimension (length) of the embedding vector
print("Dimension of the embedding vector ", len(embedding_1))                   # Typically 1536 or 2048 depending on model

In [ ]:
# Verify if both embeddings have the same dimension (should be True)
len(embedding_1) == len(embedding_2)

# Return/display the two embedding vectors for further inspection or use
embedding_1, embedding_2

### Vector Database

#### Setup Vector Database Directory

In [ ]:
out_dir = 'chroma_db'    # complete the code to define the name of the vector database

if not os.path.exists(out_dir):
  os.makedirs(out_dir)

#### Create Vector Database from Documents

In [ ]:
# Building the vector store and saving it to disk for future use
vectorstore = Chroma.from_documents(
    document_chunks,                                                            # Documents to index
    embedding_model,                                                            # Embedding model for converting text to vectors
    persist_directory=out_dir                                                   # Save vector DB files here
)

#### Load Vector Database

In [ ]:
vectorstore = Chroma(
    persist_directory=out_dir,
    embedding_function=embedding_model
)

#### Explore Vector Database and Perform Searches

In [ ]:
vectorstore.embeddings

write an instruction on what to do in next cell

In [ ]:
vectorstore.similarity_search("sepsis treatment",k=3) #Complete the code to pass a query and an appropriate k value

### Retrieval and Response Generation using Vector Search

#### Convert Vector Database into a Retriever and Retrieve Relevant Documents

In [ ]:
retriever = vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 5} #Complete the code to pass an appropriate k value
)

### System and User Prompt Template

Prompts guide the model to generate accurate responses. Here, we define two parts:

    1. The system message describing the assistant's role.
    2. A user message template including context and the question.

In [ ]:
qna_system_message = """
You are a helpful medical assistant. Use the provided context to answer the user's question accurately and concisely. If the information is not in the context, state that you don't know.
"""  #Complete the code to define the system message

In [ ]:
qna_user_message_template = """
Context: {context}
Question: {question}
Answer:
"""  #Complete the code to define the user message

### Response Function

In [ ]:
def generate_rag_response(user_input,k=3,max_tokens=128,temperature=0,top_p=0.95):
    global qna_system_message,qna_user_message_template
    # Retrieve relevant document chunks
    relevant_document_chunks = retriever.get_relevant_documents(query=user_input,k=k)
    context_list = [d.page_content for d in relevant_document_chunks]

    # Combine document chunks into a single context
    context_for_query = ". ".join(context_list)

    user_message = qna_user_message_template.replace('{context}', context_for_query)
    user_message = user_message.replace('{question}', user_input)

    # Generate the response
    try:
        response = client.chat.completions.create(
        model="gpt-3.5-turbo",   # Complete the code by specifying the model to be used.
        messages=[
            {"role": "system", "content": qna_system_message},
            {"role": "user", "content": user_message}
        ],
        max_tokens=max_tokens,
        temperature=temperature,
        top_p=top_p
        )
        # Extract and print the generated text from the response
        response = response.choices[0].message.content.strip()
    except Exception as e:
        response = f'Sorry, I encountered the following error: \n {e}'

    return response

## Question Answering using RAG

### Question 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
response_with_rag_1 = generate_rag_response(question_1)
response_with_rag_1

### Question 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
response_with_rag_2 = generate_rag_response(question_2) #Complete the code to pass the user input
response_with_rag_2

### Question 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
response_with_rag_3 = generate_rag_response(question_3) #Complete the code to pass the user input
response_with_rag_3

### Question 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
response_with_rag_4 = generate_rag_response(question_4) #Complete the code to pass the user input
response_with_rag_4

### Storing the RAG system outputs


In [ ]:
# Add the results to a new column in the DataFrame
result_df['responses_with_RAG'] = [response_with_rag_1, response_with_rag_2, response_with_rag_3, response_with_rag_4]

# Display the DataFrame
result_df.head()

## Output Evaluation

#### **Defining required System Prompts**

In [ ]:
groundedness_rater_system_message = """
You are an expert at evaluating the groundedness of a generated answer. Your task is to rate the answer based on whether it is fully supported by the provided context. Respond with a score from 1 to 5, where 1 is not at all grounded and 5 is fully grounded. Provide a brief explanation for your score.
""" #Complete the code to define the prompt to evaluate groundedness

In [ ]:
relevance_rater_system_message = """
You are an expert at evaluating the relevance of a generated answer to a given question. Your task is to rate the answer based on how directly and comprehensively it addresses the question. Respond with a score from 1 to 5, where 1 is not at all relevant and 5 is perfectly relevant. Provide a brief explanation for your score.
""" #Complete the code to define the prompt to evaluate relevance

In [ ]:
user_message_template = """
###Question
{question}

###Context
{context}

###Answer
{answer}
"""

#### **Definig the LLM-as-a-Judge Evaluation function**

In [ ]:
def generate_ground_relevance_response(user_input,response,  max_tokens=128, temperature=0, top_p=0.95):  # Complete the code to set default paramenters
    global qna_user_message_template

    context_for_query = [doc.page_content for doc in retriever.get_relevant_documents(user_input, k=5)]

    # Combine user_prompt and system_message to create the prompt
    groundedness_prompt = f"""[INST]{groundedness_rater_system_message}\n
                {'user'}: {user_message_template.format(context=context_for_query, question=user_input, answer=response)}
                [/INST]"""

    # Combine user_prompt and system_message to create the prompt
    relevance_prompt = f"""[INST]{relevance_rater_system_message}\n
                {'user'}: {user_message_template.format(context=context_for_query, question=user_input, answer=response)}
                [/INST]"""

    response_1 = client.chat.completions.create(
            model="gpt-3.5-turbo",   # Complete the code by specifying the model to be used.
            messages=[
                {"role": "user", "content": groundedness_prompt}
                ],
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p
            )

    response_2 = client.chat.completions.create(
            model="gpt-3.5-turbo",   # Complete the code by specifying the model to be used.
            messages=[
                {"role": "user", "content": relevance_prompt}
                ],
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p
            )

    return response_1.choices[0].message.content,response_2.choices[0].message.content

#### **Evaluation 1: Base Prompt Response Evaluation**

In [ ]:
# Question 1
ground,rel = generate_ground_relevance_response(user_input=result_df.questions[0], response=result_df.base_prompt_responses[0], max_tokens=516)
print(ground,end="\n\n")
print(rel)

In [ ]:
# Question 2
ground,rel = generate_ground_relevance_response(user_input=result_df.questions[1], response=result_df.base_prompt_responses[1], max_tokens=516)  #Complete the code to calculate the groundedness and relevance score
print(ground,end="\n\n")
print(rel)

In [ ]:
# Question 3
ground,rel = generate_ground_relevance_response(user_input=result_df.questions[2], response=result_df.base_prompt_responses[2], max_tokens=516)  #Complete the code to calculate the groundedness and relevance score
print(ground,end="\n\n")
print(rel)

In [ ]:
# Question 4
ground,rel = generate_ground_relevance_response(user_input=result_df.questions[3], response=result_df.base_prompt_responses[3], max_tokens=516)  #Complete the code to calculate the groundedness and relevance score
print(ground,end="\n\n")
print(rel)

#### **Evaluation 2: Prompt Engineering Response Evaluation**

In [ ]:
# Question 1
ground,rel = generate_ground_relevance_response(user_input=result_df.questions[0], response=result_df.responses_with_prompt_eng[0], max_tokens=516)
print(ground,end="\n\n")
print(rel)

In [ ]:
# Question 2
ground,rel = generate_ground_relevance_response(user_input=result_df.questions[1], response=result_df.responses_with_prompt_eng[1], max_tokens=516)  #Complete the code to calculate the groundedness and relevance score
print(ground,end="\n\n")
print(rel)

In [ ]:
# Question 3
ground,rel = generate_ground_relevance_response(user_input=result_df.questions[2], response=result_df.responses_with_prompt_eng[2], max_tokens=516)  #Complete the code to calculate the groundedness and relevance score
print(ground,end="\n\n")
print(rel)

In [ ]:
# Question 4
ground,rel = generate_ground_relevance_response(user_input=result_df.questions[3], response=result_df.responses_with_prompt_eng[3], max_tokens=516)  #Complete the code to calculate the groundedness and relevance score
print(ground,end="\n\n")
print(rel)

#### **Evaluation 3: RAG Response Evaluation**

In [ ]:
# Question 1
ground,rel = generate_ground_relevance_response(user_input=result_df.questions[0], response=result_df.responses_with_RAG[0], max_tokens=516)  #Complete the code to calculate the groundedness and relevance
print(ground,end="\n\n")
print(rel)

In [ ]:
# Question 2
ground,rel = generate_ground_relevance_response(user_input=result_df.questions[1], response=result_df.responses_with_RAG[1], max_tokens=516)  #Complete the code to calculate the groundedness and relevance
print(ground,end="\n\n")
print(rel)

In [ ]:
# Question 3
ground,rel = generate_ground_relevance_response(user_input=result_df.questions[2], response=result_df.responses_with_RAG[2], max_tokens=516)  #Complete the code to calculate the groundedness and relevance
print(ground,end="\n\n")
print(rel)

In [ ]:
# Question 4
ground,rel = generate_ground_relevance_response(user_input=result_df.questions[3], response=result_df.responses_with_RAG[3], max_tokens=516)  #Complete the code to calculate the groundedness and relevance
print(ground,end="\n\n")
print(rel)

## Actionable Insights and Business Recommendations


*   
*  
*



<font size=6 color='#4682B4'>Power Ahead</font>
___